In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import time
from functools import wraps
import plotly.express as px

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# 1. Timing Decorator for Epochs
def time_epoch(func):
  @wraps(func)
  def wrapper(*args, **kwargs):
    start_time = time.time()
    result = func(*args, **kwargs)
    epoch_time = time.time() - start_time
    print(f"{func.__name__} took {epoch_time:.4f} seconds")
    return result
  return wrapper

# 2. Data Exploration Function
def explore_data(df):
  print("Dataset Info:")
  print(df.info())
  print("\nSummary Statistics:")
  print(df.describe())
  print("\nCorrelation with Insurance Price:")
  print(df.corr()['insurance_price'].sort_values(ascending=False))
  # Visualize correlation with market price
  px.scatter(df['market_price'], df['insurance_price'])
  display(px)


def select_features(X, y, feature_names, threshold=0.4):
  corr_matrix = np.corrcoef(X.T, y.T)[:-1, -1]
  print(corr_matrix, type(corr_matrix), corr_matrix.shape)

  selected_features = np.abs(corr_matrix) > threshold
  selected_names = feature_names[selected_features]
  
  print(f"Selected features: {selected_names}")
  return X[:, selected_features], selected_names

# 3. Custom Dataset Class
class VehicleInsuranceDataset(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
 
  def __len__(self):
    return len(self.X)
 
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

# 4. Data Preprocessing Function
def preprocess_data(df, target_col='insurance_price'):
  X = df.drop(columns=[target_col]).values
  y = df[target_col].values
  scaler = StandardScaler()
  X_scaled = scaler.fit_transform(X)
  return X_scaled, y, scaler

# 5. Data Splitting Function
def split_data(dataset, train_ratio=0.7, val_ratio=0.15):
  train_size = int(train_ratio * len(dataset))
  val_size = int(val_ratio * len(dataset))
  test_size = len(dataset) - train_size - val_size
  train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
  return train_dataset, val_dataset, test_dataset

# 6. Data Loader Function
def create_data_loaders(train_dataset, val_dataset, test_dataset, batch_size=32):
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_dataset, batch_size=batch_size)
  test_loader = DataLoader(test_dataset, batch_size=batch_size)
  return train_loader, val_loader, test_loader

# 7. Regression Model
class RegressionModel(nn.Module):
  def __init__(self, input_dim):
    super(RegressionModel, self).__init__()
    self.layers = nn.Sequential(
      nn.Linear(input_dim, 1),
    )
 
  def forward(self, x):
    return self.layers(x)

# 10. Training Function with Early Stopping
@time_epoch
def train_epoch(model, train_loader, criterion, optimizer):
  model.train()
  epoch_train_loss = 0
  for X_batch, y_batch in train_loader:
    optimizer.zero_grad()
    y_pred = model(X_batch)
    loss = criterion(y_pred, y_batch)
    loss.backward()
    optimizer.step()
    epoch_train_loss += loss.item() * X_batch.size(0)
  return epoch_train_loss / len(train_loader.dataset)

@time_epoch
def validate_epoch(model, val_loader, criterion):
  model.eval()
  epoch_val_loss = 0
  with torch.no_grad():
    for X_batch, y_batch in val_loader:
      y_pred = model(X_batch)
      loss = criterion(y_pred, y_batch)
      epoch_val_loss += loss.item() * X_batch.size(0)
  return epoch_val_loss / len(val_loader.dataset)

def train_model(model, train_loader, val_loader, epochs=100, patience=5):
  criterion = nn.MSELoss()
  optimizer = optim.Adam(model.parameters(), lr=0.001)
  train_losses = []
  val_losses = []
  best_val_loss = float('inf')
  patience_counter = 0
 
  for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss = validate_epoch(model, val_loader, criterion)
   
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
   
    # Early Stopping
    if val_loss < best_val_loss:
      best_val_loss = val_loss
      patience_counter = 0
      torch.save(model.state_dict(), 'best_model.pth')
    else:
      patience_counter += 1
      if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break
 
  return train_losses, val_losses

# 11. Evaluation Function
def evaluate_model(model, test_loader, criterion):
  model.eval()
  test_loss = 0
  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      y_pred = model(X_batch)
      loss = criterion(y_pred, y_batch)
      test_loss += loss.item() * X_batch.size(0)
  test_loss /= len(test_loader.dataset)
  print(f"Test Loss: {test_loss:.4f}")
  return test_loss

# 12. Main Execution
def main():
  # Generate and explore data
  df = pd.read_csv(r"F:\KPIT\PyTorch\Datasets\vehicle_insurance_data.csv")
  explore_data(df)
 
  # Preprocess data
  X_scaled, y, scaler = preprocess_data(df)
 
  # Feature selection
  feature_names = np.array(df.drop(columns=['insurance_price']).columns)
  X_selected, selected_features = select_features(X_scaled, y, feature_names)
 
  # Create dataset
  dataset = VehicleInsuranceDataset(X_selected, y)
 
  # Split data
  train_dataset, val_dataset, test_dataset = split_data(dataset)
 
  # Create data loaders
  train_loader, val_loader, test_loader = create_data_loaders(train_dataset, val_dataset, test_dataset)
 
  # Initialize model
  input_dim = X_selected.shape[1]
  model = RegressionModel(input_dim)
 
  # Train model
  train_losses, val_losses = train_model(model, train_loader, val_loader)
 
  # Evaluate model
  model.load_state_dict(torch.load('best_model.pth'))
  criterion = nn.MSELoss()
  evaluate_model(model, test_loader, criterion)

if __name__ == "__main__":
  main()

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   market_price     1000 non-null   float64
 1   vehicle_age      1000 non-null   int64  
 2   mileage          1000 non-null   float64
 3   horsepower       1000 non-null   int64  
 4   fuel_efficiency  1000 non-null   float64
 5   safety_rating    1000 non-null   float64
 6   insurance_price  1000 non-null   float64
dtypes: float64(5), int64(2)
memory usage: 54.8 KB
None

Summary Statistics:
       market_price  vehicle_age        mileage   horsepower  fuel_efficiency  \
count   1000.000000  1000.000000    1000.000000  1000.000000      1000.000000   
mean   54123.089799     7.671000   74043.582776   244.457000        27.165207   
std    26292.362575     4.079387   43174.322342    84.968581         7.180115   
min    10416.882070     1.000000    1028.071534   100.000000      

<module 'plotly.express' from 'c:\\Users\\Pragyan Prakhar\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\plotly\\express\\__init__.py'>

[ 0.77211268  0.22415887  0.27747907  0.4745224  -0.25627432  0.13245872] <class 'numpy.ndarray'> (6,)
Selected features: ['market_price' 'horsepower']
train_epoch took 0.1403 seconds
validate_epoch took 0.0084 seconds
Epoch 1, Train Loss: 37710497.7371, Val Loss: 39095078.0267
train_epoch took 0.0791 seconds
validate_epoch took 0.0069 seconds
Epoch 2, Train Loss: 37710177.7371, Val Loss: 39094731.6800
train_epoch took 0.0798 seconds
validate_epoch took 0.0043 seconds
Epoch 3, Train Loss: 37709852.7543, Val Loss: 39094405.3867
train_epoch took 0.0890 seconds
validate_epoch took 0.0115 seconds
Epoch 4, Train Loss: 37709536.9371, Val Loss: 39094061.9200
train_epoch took 0.0800 seconds
validate_epoch took 0.0075 seconds
Epoch 5, Train Loss: 37709212.4114, Val Loss: 39093726.4533
train_epoch took 0.0914 seconds
validate_epoch took 0.0087 seconds
Epoch 6, Train Loss: 37708893.8971, Val Loss: 39093392.4267
train_epoch took 0.0818 seconds
validate_epoch took 0.0030 seconds
Epoch 7, Train Loss